# DFlash与DSpark数值案例

用小矩阵把一轮投机解码数据流跑通（不加载真实LLM），内容：

* 主模型与草稿模型每一步交换了什么
* Draft里$H_{ctx}$如何进K/V、mask位一次算出什么
* DSpark多出来的Markov / confidence / 按$\mathrm{SPS}$裁剪验证，分别得到什么数

相关文章链接：[快速理解并行投机解码(DFlash/DSpark)](https://zhuanlan.zhihu.com/p/2069029506447417522)

Author: kaiyuan

Email: kaiyuanxie@yeah.net


# 1 背景速览

两个计算模式：

* 词语接龙：causal mask，自回归挨个吐字（主模型decoding、以及EAGLE这类自回归草稿）
* 完型填空：可读全句、一次填多空（DFlash草稿的并行block）

投机解码：小模型先猜一截，主模型再校验。DFlash把草稿改成完型填空，并用主模型中间层融成$H_{ctx}$注入Draft；DSpark再在骨干后加串行头与Hardware-Aware Prefix Scheduler。

下面从第2节起逐步做数值演示。


# 2 准备工作

初始化词表与随机权重。词表用“我是”+字母位，方便人眼读序列。


In [1]:
import math
import numpy as np

# 词表：前缀“我是”+后续英文字母位
VOCAB = ["<pad>", "<mask>", "我", "是", "k", "a", "i", "y"]
MASK_ID = 1
VOCAB_SIZE = len(VOCAB)
D_MODEL = 4
N_DRAFT_LAYERS = 2
BLOCK_SIZE = 4
GAMMA = BLOCK_SIZE - 1  # DFlash默认只预测mask位
MARKOV_RANK = 2
CTX_LAYER_IDS = [1, 2]

rng = np.random.default_rng(42)

def softmax(x, axis=-1):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def rms_norm(x, eps=1e-6):
    return x / np.sqrt(np.mean(x ** 2, axis=-1, keepdims=True) + eps)

def tokens_to_str(ids):
    return "".join(VOCAB[i] for i in ids)

def show(name, arr, meaning):
    a = np.asarray(arr)
    print(f"[{name}] shape={a.shape}  # {meaning}")
    print(np.round(a, 4))
    print()

def init_linear(in_f, out_f, scale=0.3):
    return rng.normal(0, scale, size=(out_f, in_f))

# Embedding / LM head 与主模型共享（演示里用同一组矩阵）
E = rng.normal(0, 0.4, size=(VOCAB_SIZE, D_MODEL))
E[MASK_ID] = rng.normal(0, 0.05, size=(D_MODEL,))
W_lm = E.copy()
W_c = init_linear(len(CTX_LAYER_IDS) * D_MODEL, D_MODEL, 0.25)

draft_layers = []
for _ in range(N_DRAFT_LAYERS):
    draft_layers.append({
        "Wq": init_linear(D_MODEL, D_MODEL),
        "Wk": init_linear(D_MODEL, D_MODEL),
        "Wv": init_linear(D_MODEL, D_MODEL),
        "Wo": init_linear(D_MODEL, D_MODEL),
        "W1": init_linear(D_MODEL, D_MODEL * 2),
        "W2": init_linear(D_MODEL * 2, D_MODEL),
    })

_TARGET_TRANSFORMS = [init_linear(D_MODEL, D_MODEL, 0.2) for _ in range(4)]
W_markov_emb = rng.normal(0, 0.3, size=(VOCAB_SIZE, MARKOV_RANK))
W_markov_proj = rng.normal(0, 0.3, size=(MARKOV_RANK, VOCAB_SIZE))
w_conf = rng.normal(0, 0.4, size=(D_MODEL + MARKOV_RANK,))

print(f"就绪 | vocab={VOCAB}")
print(f"D={D_MODEL}, draft_layers={N_DRAFT_LAYERS}, block_size={BLOCK_SIZE}, gamma={GAMMA}")


就绪 | vocab=['<pad>', '<mask>', '我', '是', 'k', 'a', 'i', 'y']
D=4, draft_layers=2, block_size=4, gamma=3


In [2]:
def target_forward(token_ids):
    """模拟主模型forward：返回末位置logits + 各层hidden。"""
    h = E[np.asarray(token_ids)].copy()
    hiddens = []
    for li, Wt in enumerate(_TARGET_TRANSFORMS):
        h = rms_norm(h + 0.15 * (li + 1) * np.tanh(h @ Wt.T))
        hiddens.append(h.copy())
    logits = h[-1] @ W_lm.T
    return logits, hiddens

def fuse_target_context(hiddens, positions=None):
    """H_ctx = RMSNorm(W_c [H^{l1}; ...; H^{lm}])"""
    if positions is None:
        positions = list(range(hiddens[0].shape[0]))
    feats = [np.concatenate([hiddens[li][p] for li in CTX_LAYER_IDS], axis=-1) for p in positions]
    return rms_norm(np.stack(feats, axis=0) @ W_c.T)

def bidirectional_attn(Q, K, V):
    scale = 1.0 / math.sqrt(Q.shape[-1])
    weights = softmax((Q @ K.T) * scale, axis=-1)
    return weights @ V, weights

def draft_layer_forward(H_d, H_ctx, layer):
    """KV injection: K/V=[proj(H_ctx); proj(H_d)], Q=proj(H_d)"""
    Q = H_d @ layer["Wq"].T
    K = np.concatenate([H_ctx @ layer["Wk"].T, H_d @ layer["Wk"].T], axis=0)
    V = np.concatenate([H_ctx @ layer["Wv"].T, H_d @ layer["Wv"].T], axis=0)
    out, attn_w = bidirectional_attn(Q, K, V)
    H = rms_norm(H_d + out @ layer["Wo"].T)
    ff = np.maximum(0, H @ layer["W1"].T) @ layer["W2"].T
    H = rms_norm(H + ff)
    return H, attn_w, (Q.shape, K.shape, V.shape)

print("已定义 target_forward / fuse_target_context / draft_layer_forward")


已定义 target_forward / fuse_target_context / draft_layer_forward


# 3 DFlash一轮

拆成Step A/B/C：

1. 主模型根据前缀写出anchor，并抽出中间层hidden融成$H_{ctx}$
2. Draft输入$[\mathrm{anchor},\langle m\rangle,\ldots]$，Attention里K/V为$[H_{ctx}\,||\,H_d]$，Query只来自$H_d$；一次forward填完mask
3. 主模型校验草稿最长前缀，再写出新的bonus

## 3.1 Step A：anchor与$H_{ctx}$


In [3]:
# Step A：主模型产出 anchor 与 H_ctx
prompt = [2, 3]  # 我是
print(f"前缀: {prompt} -> {tokens_to_str(prompt)}")

logits, hiddens = target_forward(prompt)
show("target_logits", logits, "末位置词表打分")
bonus = int(np.argmax(logits))
print(f"argmax得到anchor/bonus = {bonus} ({VOCAB[bonus]})")
print()

for li in CTX_LAYER_IDS:
    show(f"H_layer_{li}", hiddens[li], "主模型中间层hidden")

H_ctx = fuse_target_context(hiddens)
show("H_ctx", H_ctx, "跨层拼接+投影+RMSNorm后的上下文，稍后注入Draft的K/V")

prefix = prompt + [bonus]
print(f"更新前缀: {prefix} -> {tokens_to_str(prefix)}")


前缀: [2, 3] -> 我是
[target_logits] shape=(8,)  # 末位置词表打分
[-0.6551 -0.0853 -0.4317  1.1611 -0.1015  0.2293 -0.129   1.2854]

argmax得到anchor/bonus = 7 (y)

[H_layer_1] shape=(2, 4)  # 主模型中间层hidden
[[-0.083  -0.9978  1.2457  1.2025]
 [ 0.0965  1.4313  0.7725 -1.1599]]

[H_layer_2] shape=(2, 4)  # 主模型中间层hidden
[[-0.2265 -1.0761  1.021   1.3222]
 [ 0.0788  1.4579  0.8699 -1.0544]]

[H_ctx] shape=(2, 4)  # 跨层拼接+投影+RMSNorm后的上下文，稍后注入Draft的K/V
[[-0.3479  1.352  -1.3751 -0.4002]
 [-1.9572  0.3613  0.1239  0.1525]]

更新前缀: [2, 3, 7] -> 我是y


## 3.2 Step B：Draft并行填空

打印内容：

* `H_d_input`：只有anchor是真token，后面是mask
* 每层`K/V`长度 = 上下文长度 + block长度，即把$H_{ctx}$拼进K/V前半段
* `mask_logits`：一次得到，位置之间没有词语接龙依赖


In [4]:
# Step B：草稿模型一次并行填空（完型填空，不是词语接龙）
print(f"anchor={bonus}({VOCAB[bonus]}), gamma={GAMMA}, mask={MASK_ID}")
print()

H_ctx_last = H_ctx[-1:]
show("H_ctx_for_draft", H_ctx_last, "本轮注入Draft的上下文（演示取末位置）")

input_ids = [bonus] + [MASK_ID] * GAMMA
H_d = E[np.asarray(input_ids)].copy()
print(f"Draft输入: {input_ids} -> {tokens_to_str(input_ids)}")
show("H_d_input", H_d, "第0位是干净anchor，后面是mask占位")

for li, layer in enumerate(draft_layers):
    H_d, attn_w, (qshape, kshape, vshape) = draft_layer_forward(H_d, H_ctx_last, layer)
    print(f"--- Draft Layer {li} ---")
    print(f"Q={qshape}（只来自Draft）; K/V={kshape}（前半H_ctx，后半H_d）")
    show(f"attn_L{li}", attn_w, "Draft query对[H_ctx|H_d]的注意力（无causal mask）")
    show(f"H_d_L{li}", H_d, "该层输出hidden")

logits_all = H_d @ W_lm.T
mask_logits = logits_all[1:]
base_probs = softmax(mask_logits, axis=-1)
draft_ids = [int(np.argmax(base_probs[k])) for k in range(GAMMA)]

show("mask_logits", mask_logits, "同一次forward得到的gamma个并行预测")
print("草稿token:")
for k, tid in enumerate(draft_ids):
    print(f"  pos{k+1}: {tid}({VOCAB[tid]}) p={base_probs[k][tid]:.4f}")
print(f"draft = {draft_ids} -> {tokens_to_str(draft_ids)}")


anchor=7(y), gamma=3, mask=1

[H_ctx_for_draft] shape=(1, 4)  # 本轮注入Draft的上下文（演示取末位置）
[[-1.9572  0.3613  0.1239  0.1525]]

Draft输入: [7, 1, 1, 1] -> y<mask><mask><mask>
[H_d_input] shape=(4, 4)  # 第0位是干净anchor，后面是mask占位
[[ 0.1651  0.1723  0.8567 -0.1626]
 [-0.0256 -0.0407  0.0308  0.0564]
 [-0.0256 -0.0407  0.0308  0.0564]
 [-0.0256 -0.0407  0.0308  0.0564]]

--- Draft Layer 0 ---
Q=(4, 4)（只来自Draft）; K/V=(5, 4)（前半H_ctx，后半H_d）
[attn_L0] shape=(4, 5)  # Draft query对[H_ctx|H_d]的注意力（无causal mask）
[[0.1846 0.2019 0.2045 0.2045 0.2045]
 [0.1973 0.2013 0.2005 0.2005 0.2005]
 [0.1973 0.2013 0.2005 0.2005 0.2005]
 [0.1973 0.2013 0.2005 0.2005 0.2005]]

[H_d_L0] shape=(4, 4)  # 该层输出hidden
[[ 0.2298  0.2692  1.9323 -0.3753]
 [-1.2054 -0.9521  0.3841  1.2218]
 [-1.2054 -0.9521  0.3841  1.2218]
 [-1.2054 -0.9521  0.3841  1.2218]]

--- Draft Layer 1 ---
Q=(4, 4)（只来自Draft）; K/V=(5, 4)（前半H_ctx，后半H_d）
[attn_L1] shape=(4, 5)  # Draft query对[H_ctx|H_d]的注意力（无causal mask）
[[0.1551 0.2136 0.2104 0.2104 0.210

## 3.3 Step C：主模型校验并写bonus

与常规投机推理相同：从左核对到第一处不一致为止；该位起草稿作废，由主模型给出bonus（下轮的anchor）。


In [5]:
# Step C：主模型校验（greedy最长前缀）+ 写出新bonus
print(f"校验前前缀: {tokens_to_str(prefix)}")
print(f"待校验草稿: {tokens_to_str(draft_ids)}")
print()

accepted = []
cur = list(prefix)
bonus_new = None
for i, d in enumerate(draft_ids):
    logits_v, _ = target_forward(cur)
    probs_v = softmax(logits_v)
    pred = int(np.argmax(logits_v))
    ok = pred == d
    print(
        f"位{i}: draft={d}({VOCAB[d]}) target={pred}({VOCAB[pred]}) "
        f"p_t={probs_v[d]:.4f}  {'ACCEPT' if ok else 'REJECT'}"
    )
    if not ok:
        bonus_new = pred
        print(f"  从该位起丢弃；主模型写出bonus={bonus_new}({VOCAB[bonus_new]})")
        break
    accepted.append(d)
    cur.append(d)
else:
    logits_v, _ = target_forward(cur)
    bonus_new = int(np.argmax(logits_v))
    print(f"全部接受；主模型再写bonus={bonus_new}({VOCAB[bonus_new]})")

new_prefix = prefix + accepted + [bonus_new]
print()
print(f"接受草稿长度={len(accepted)}/{len(draft_ids)}（另外写入1个bonus）")
print(f"本轮后序列: {tokens_to_str(new_prefix)}")


校验前前缀: 我是y
待校验草稿: 我我我

位0: draft=2(我) target=7(y) p_t=0.0834  REJECT
  从该位起丢弃；主模型写出bonus=7(y)

接受草稿长度=0/3（另外写入1个bonus）
本轮后序列: 我是yy


# 4 DSpark：在DFlash骨干上多两段

parallel block仍是DFlash式完型填空；之后加sequential block（Markov/RNN）与筛选。

## 4.1 Step D：并行logits + 串行修正 + confidence

先一次算出各位置$U$，再从左到右用上一token的转移偏置$B$修正后采样，并估计条件置信度$c_k$。


In [6]:
# Step D：DSpark = 并行骨干 + Markov串行头 + confidence
print("D-1 并行骨干：一次forward得到各位置base logits U")

def markov_bias(prev_id):
    return W_markov_emb[prev_id] @ W_markov_proj

gamma_ds = BLOCK_SIZE
input_ids_ds = [bonus] + [MASK_ID] * (gamma_ds - 1)
H_d = E[np.asarray(input_ids_ds)].copy()
for layer in draft_layers:
    H_d, _, _ = draft_layer_forward(H_d, H_ctx_last, layer)

U = H_d @ W_lm.T
show("U", U, "并行骨干打分（此时块内还没串起来）")

print("D-2 从左到右：U + Markov偏置B，采样并打confidence")
draft_ids_ds, confs = [], []
prev = bonus
for k in range(gamma_ds):
    B = markov_bias(prev)
    logits_k = U[k] + B
    probs_k = softmax(logits_k)
    tid = int(np.argmax(probs_k))
    feat = np.concatenate([H_d[k], W_markov_emb[prev]])
    c = 1.0 / (1.0 + math.exp(-float(w_conf @ feat)))
    print(f"k={k+1} prev={prev}({VOCAB[prev]}) -> {tid}({VOCAB[tid]}) p={probs_k[tid]:.4f} c={c:.4f}")
    draft_ids_ds.append(tid)
    confs.append(c)
    prev = tid

print(f"草稿: {tokens_to_str(draft_ids_ds)}")
print(f"confidence: {[round(c, 4) for c in confs]}")


D-1 并行骨干：一次forward得到各位置base logits U
[U] shape=(4, 8)  # 并行骨干打分（此时块内还没串起来）
[[ 0.449   0.0403  0.5761  0.4998  0.6865  0.9769  0.3663  1.7476]
 [ 0.8651  0.1535  0.9123 -0.6662  0.3508  0.5692  0.642   0.1   ]
 [ 0.8651  0.1535  0.9123 -0.6662  0.3508  0.5692  0.642   0.1   ]
 [ 0.8651  0.1535  0.9123 -0.6662  0.3508  0.5692  0.642   0.1   ]]

D-2 从左到右：U + Markov偏置B，采样并打confidence
k=1 prev=7(y) -> 7(y) p=0.2872 c=0.4143
k=2 prev=7(y) -> 0(<pad>) p=0.2110 c=0.3149
k=3 prev=0(<pad>) -> 0(<pad>) p=0.2166 c=0.3373
k=4 prev=0(<pad>) -> 0(<pad>) p=0.2166 c=0.3373
草稿: y<pad><pad><pad>
confidence: [0.4143, 0.3149, 0.3373, 0.3373]


## 4.2 Step E：按“值不值得验”裁剪长度

Hardware-Aware Prefix Scheduler大致是：

* $a_j=\prod_{i\le j}c_i$
* 用profile好的$\mathrm{SPS}(B)$估$\Theta=\tau\cdot\mathrm{SPS}(B)$
* 贪心加长验证前缀，吞吐不再升就停，只把前$\ell^*$个送给主模型


In [7]:
# Step E：Hardware-Aware Prefix Scheduler
print("E-1 前缀存活概率 a_j = Π c_i")
SPS = {1: 100.0, 2: 95.0, 3: 88.0, 4: 78.0, 5: 65.0, 6: 50.0, 7: 38.0, 8: 28.0}
a, prod = [], 1.0
for c in confs:
    prod *= c
    a.append(prod)
print(f"c={ [round(x,4) for x in confs] }")
print(f"a={ [round(x,4) for x in a] }")

print("E-2 按SPS(B)扩展验证长度，Theta不再升就停")
R = 1
best_theta = R * SPS[R]
best_ell, tau, Bsz = 0, float(R), R
for j, aj in enumerate(a, start=1):
    B_new = Bsz + 1
    tau_new = tau + aj
    theta = tau_new * SPS.get(B_new, SPS[max(SPS)])
    print(f"  ell={j}: B={B_new}, 增量a={aj:.4f}, Theta={theta:.2f}")
    if theta > best_theta:
        best_theta, best_ell, tau, Bsz = theta, j, tau_new, B_new
    else:
        print(f"  停在 ell*={best_ell}")
        break

scheduled = draft_ids_ds[:best_ell]
print(f"ell*={best_ell}, 送去校验: {tokens_to_str(scheduled)}")

print("E-3 主模型只校验该前缀")
accepted_ds, cur = [], list(prefix)
bonus_ds = None
for i, d in enumerate(scheduled):
    logits_v, _ = target_forward(cur)
    pred = int(np.argmax(logits_v))
    ok = pred == d
    print(f"  位{i}: draft={d}({VOCAB[d]}) target={pred}({VOCAB[pred]}) {'ACCEPT' if ok else 'REJECT'}")
    if not ok:
        bonus_ds = pred
        break
    accepted_ds.append(d)
    cur.append(d)
else:
    logits_v, _ = target_forward(cur if scheduled else prefix)
    bonus_ds = int(np.argmax(logits_v))

print(f"结果: accepted={tokens_to_str(accepted_ds)}, bonus={bonus_ds}({VOCAB[bonus_ds]})")


E-1 前缀存活概率 a_j = Π c_i
c=[0.4143, 0.3149, 0.3373, 0.3373]
a=[0.4143, 0.1305, 0.044, 0.0148]
E-2 按SPS(B)扩展验证长度，Theta不再升就停
  ell=1: B=2, 增量a=0.4143, Theta=134.36
  ell=2: B=3, 增量a=0.1305, Theta=135.94
  ell=3: B=4, 增量a=0.0440, Theta=123.92
  停在 ell*=2
ell*=2, 送去校验: y<pad>
E-3 主模型只校验该前缀
  位0: draft=7(y) target=7(y) ACCEPT
  位1: draft=0(<pad>) target=7(y) REJECT
结果: accepted=y, bonus=7(y)


# 5 符号速查

| 符号 | 含义 |
|---|---|
| Target / Draft | 主模型 / 草稿模型 |
| anchor / bonus | 主模型写出的干净token：进本轮输出，并作下轮Draft条件 |
| $H_{ctx}$ | 主模型多层hidden融合后的上下文 |
| KV injection | 把$H_{ctx}$拼进Draft每层K/V |
| $\gamma$ | 草稿token数；DFlash常为block_size-1 |
| $U$ / $B$ | 并行骨干logits / Markov转移偏置 |
| $c$ / $a$ / $\ell^*$ | 条件置信度 / 前缀存活概率 / 实际验证长度 |
| $\mathrm{SPS}(B)$ | 引擎在batch大小为$B$时的Steps Per Second |
